# Decoder-only Transformer (GPT-style) from Scratch

The third model: a small **GPT-style decoder-only transformer**, written from scratch in PyTorch.

It uses **exactly the same data as the LSTM** (`vocab.pkl` and `encoded_*.pkl` from notebook 01), so the
comparison shows the effect of the **architecture**:

| Model | Architecture | Pretrained? |
|---|---|---|
| LSTM (03) | encoder-decoder RNN | no |
| **GPT (06)** | **decoder-only transformer** | **no** |
| CodeT5 (04–05) | encoder-decoder transformer | yes |

**Difference from seq2seq:** there is no separate encoder. Context and completion form one sequence,
and the model simply predicts the **next token** at every position:

```
<BOS> def add ( a , b ) : <NEWLINE> <INDENT> | return a + b <NEWLINE> <EOS>
           ^ context (32 tokens)                ^ completion (what we train on)
```

In [ ]:
import math
import pickle
import random

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset
from torch.nn.utils.rnn import pad_sequence
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

In [ ]:
SEQ_LEN = 32           # same as LSTM
MAX_OUTPUT_LEN = 50    # same as LSTM
STRIDE = 16            # same as LSTM
MAX_LEN = 1 + SEQ_LEN + MAX_OUTPUT_LEN + 1   # <BOS> + context + target + <EOS>

N_TRAIN = 500_000      # random training chunks per run (full set is ~3M)
N_GEN_EVAL = 2_000     # test examples used for generation metrics

D_MODEL = 256
N_LAYERS = 4
N_HEADS = 4
DROPOUT = 0.1

BATCH_SIZE = 64
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 10
PATIENCE = 3

BEST_MODEL_PATH = "best_gpt_code_completion.pt"

random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Load encoded data (output of notebook 01)

In [ ]:
with open('/content/encoded_train.pkl','rb') as f:
  encoded_train=pickle.load(f)

with open('/content/encoded_test.pkl','rb') as f:
  encoded_test=pickle.load(f)

with open('/content/encoded_valid.pkl','rb') as f:
  encoded_valid=pickle.load(f)

with open('/content/vocab.pkl','rb') as f:
  vocab=pickle.load(f)

vocab_size = len(vocab)
id_to_token = {i: t for t, i in vocab.items()}

PAD_IDX = vocab["<PAD>"]
BOS_IDX = vocab["<BOS>"]
EOS_IDX = vocab["<EOS>"]
IGNORE_IDX = -100

## Dataset

Same split positions as `DatasetPrep` in the LSTM notebook. The difference is how one example looks:

- `sequence = <BOS> + context + target + <EOS>`
- `x = sequence[:-1]`, `y = sequence[1:]` → at every position the label is the next token
- labels for the **context** part are set to `-100`, so the loss is only on the **completion**
  (the same thing the LSTM decoder is trained to predict)

In [ ]:
class GPTDataset(Dataset):

    def __init__(self, data, vocab,
                 seq_length=SEQ_LEN,
                 max_output_length=MAX_OUTPUT_LEN,
                 stride=STRIDE):

        self.data = data
        self.vocab = vocab
        self.seq_length = seq_length
        self.max_output_length = max_output_length

        # Each element = (row_number, split_position)
        self.indices = []

        for row_idx, row in enumerate(data):
            for i in range(seq_length, len(row), stride):
                self.indices.append((row_idx, i))

    def __getitem__(self, idx):

        row_idx, i = self.indices[idx]
        row = self.data[row_idx]

        context = row[i-self.seq_length:i]

        # ---------- Find newline ----------
        nearest_line_end = len(row)

        for j in range(i, len(row)):
            if row[j] == self.vocab["<NEWLINE>"]:
                nearest_line_end = j
                break

        end = min(i + self.max_output_length,
                  nearest_line_end + 1)

        target = row[i:end]

        # ---------- One sequence ----------
        sequence = [BOS_IDX] + context + target + [EOS_IDX]

        x = sequence[:-1]
        y = sequence[1:]

        # don't train on predicting the context
        y = [IGNORE_IDX] * len(context) + y[len(context):]

        return torch.tensor(x), torch.tensor(y)

    def __len__(self):
        return len(self.indices)

Padding goes on the **right**. Thanks to the causal mask, real tokens never look at the padding after them,
and padded labels are `-100`, so padding has no effect on the loss.

In [ ]:
def collate_fn(batch):
    inputs, labels = zip(*batch)

    inputs = pad_sequence(inputs, batch_first=True, padding_value=PAD_IDX)
    labels = pad_sequence(labels, batch_first=True, padding_value=IGNORE_IDX)

    return inputs, labels

In [ ]:
train_dataset = GPTDataset(encoded_train, vocab)
valid_dataset = GPTDataset(encoded_valid, vocab)
test_dataset = GPTDataset(encoded_test, vocab)

print(len(train_dataset), len(valid_dataset), len(test_dataset))

# random subset of training chunks to keep an epoch short on Colab
train_subset = Subset(
    train_dataset,
    random.sample(range(len(train_dataset)), min(N_TRAIN, len(train_dataset)))
)

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

In [ ]:
x, y = next(iter(train_loader))

print(x.shape)
print(y.shape)

## Model

### Causal self-attention

Every token builds a **query**, **key** and **value** vector.

- `scores = Q · Kᵀ / √d` — how much each token should attend to every other token
- **causal mask** — token *t* may only look at tokens `≤ t` (it cannot see the future it has to predict)
- `softmax(scores) · V` — weighted mix of the values

With several **heads**, this is done in parallel on smaller slices of the embedding, so each head can focus
on a different relation (brackets, variable names, indentation, ...).

In [ ]:
class CausalSelfAttention(nn.Module):

    def __init__(self, d_model, n_heads, max_len, dropout):
        super().__init__()
        assert d_model % n_heads == 0

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.query = nn.Linear(d_model, d_model)
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)
        self.proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

        # lower-triangular matrix: position t can see positions 0..t
        mask = torch.tril(torch.ones(max_len, max_len)).bool()
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape

        # (B, T, C) -> (B, n_heads, T, head_dim)
        q = self.query(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.key(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.value(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # (B, n_heads, T, T)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(~self.mask[:T, :T], float("-inf"))

        weights = torch.softmax(scores, dim=-1)
        weights = self.dropout(weights)

        out = weights @ v                                  # (B, n_heads, T, head_dim)
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.proj(out)

### Transformer block

`attention → feed-forward`, each with a **LayerNorm** before it and a **residual connection** around it
(the "pre-norm" layout used by GPT-2, which trains more stably).

In [ ]:
class FeedForward(nn.Module):

    def __init__(self, d_model, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):

    def __init__(self, d_model, n_heads, max_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, max_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

### GPT

- **token embedding + position embedding** (attention by itself has no notion of order)
- a stack of blocks
- final LayerNorm + linear layer to vocabulary logits
- **weight tying:** the output layer reuses the token embedding matrix (fewer parameters, usually better)

`generate` does greedy decoding: predict the next token, append it, repeat until `<EOS>`.

In [ ]:
class GPT(nn.Module):

    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len, dropout):
        super().__init__()
        self.max_len = max_len

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)

        self.blocks = nn.ModuleList(
            [Block(d_model, n_heads, max_len, dropout) for _ in range(n_layers)]
        )

        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        # weight tying
        self.lm_head.weight = self.token_embedding.weight

    def forward(self, idx):
        B, T = idx.shape

        positions = torch.arange(T, device=idx.device)
        x = self.token_embedding(idx) + self.position_embedding(positions)
        x = self.dropout(x)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        return self.lm_head(x)                             # (B, T, vocab_size)

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, eos_idx):
        # idx: (1, T) prompt token ids
        for _ in range(max_new_tokens):
            logits = self(idx[:, -self.max_len:])
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)

            idx = torch.cat([idx, next_token], dim=1)

            if next_token.item() == eos_idx:
                break

        return idx

In [ ]:
model = GPT(vocab_size, D_MODEL, N_LAYERS, N_HEADS, MAX_LEN, DROPOUT).to(device)

print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_IDX)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1
)

## Training

Token accuracy is measured on the completion tokens only (labels `!= -100`), like the LSTM.

In [ ]:
def run_epoch(model, loader, criterion, device, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_tokens = 0

    with torch.set_grad_enabled(training):
        for x, y in tqdm(loader, leave=False):

            x = x.to(device)
            y = y.to(device)

            logits = model(x)

            loss = criterion(logits.reshape(-1, logits.shape[-1]), y.reshape(-1))

            if training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item()

            mask = y != IGNORE_IDX
            predictions = logits.argmax(dim=-1)
            total_correct += ((predictions == y) & mask).sum().item()
            total_tokens += mask.sum().item()

    return total_loss / len(loader), total_correct / total_tokens


def train_one_epoch(model, loader, optimizer, criterion, device):
    return run_epoch(model, loader, criterion, device, optimizer)


def validate(model, loader, criterion, device):
    return run_epoch(model, loader, criterion, device)

In [ ]:
best_val_loss = float("inf")
epochs_without_improvement = 0

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

for epoch in range(NUM_EPOCHS):

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(model, valid_loader, criterion, device)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(
        f"Epoch [{epoch+1:02d}/{NUM_EPOCHS}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Val Loss: {val_loss:.4f} "
        f"| Train Acc: {train_acc*100:.2f}% "
        f"| Val Acc: {val_acc*100:.2f}% "
        f"| LR: {current_lr:.6f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "best_val_loss": best_val_loss,
            },
            BEST_MODEL_PATH
        )
        print("   ✓ Best model saved")

    else:
        epochs_without_improvement += 1
        print(f"   No improvement ({epochs_without_improvement}/{PATIENCE})")

        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered.")
            break

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GPT Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(train_accuracies, label="Training Accuracy")
plt.plot(val_accuracies, label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Token Accuracy")
plt.title("GPT Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

## Evaluation

- **Test loss / perplexity / token accuracy** — teacher-forced, on the same vocab as the LSTM, so directly comparable.
- **Generation metrics** — the model writes the whole line by itself (greedy), compared token by token with
  the true line:
    - **Generation token accuracy:** fraction of target tokens predicted correctly at the right position
    - **Exact line match:** fraction of lines generated completely correctly

In [ ]:
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

test_loss, test_accuracy = validate(model, test_loader, criterion, device)

print(f"Test Loss       : {test_loss:.4f}")
print(f"Test Perplexity : {math.exp(test_loss):.2f}")
print(f"Test Accuracy   : {test_accuracy * 100:.2f}%")

In [ ]:
def generation_metrics(model, dataset, n_samples):
    model.eval()

    correct_tokens = 0
    total_tokens = 0
    exact_matches = 0

    for idx in random.sample(range(len(dataset)), n_samples):
        x, y = dataset[idx]

        prompt = x[:SEQ_LEN + 1]                           # <BOS> + context
        target = y[y != IGNORE_IDX].tolist()[:-1]          # completion without <EOS>

        generated = model.generate(
            prompt.unsqueeze(0).to(device),
            max_new_tokens=MAX_OUTPUT_LEN + 1,
            eos_idx=EOS_IDX
        )[0, len(prompt):].tolist()

        if generated and generated[-1] == EOS_IDX:
            generated = generated[:-1]

        correct_tokens += sum(p == t for p, t in zip(generated, target))
        total_tokens += len(target)
        exact_matches += generated == target

    return correct_tokens / total_tokens, exact_matches / n_samples

In [ ]:
gen_token_accuracy, exact_match = generation_metrics(model, test_dataset, N_GEN_EVAL)

print("\n===== FINAL GPT TEST RESULTS =====")
print(f"Test Loss              : {test_loss:.4f}")
print(f"Test Accuracy          : {test_accuracy * 100:.2f}%")
print(f"Generation Token Acc   : {gen_token_accuracy * 100:.2f}%")
print(f"Exact Line Match       : {exact_match * 100:.2f}%")

## Autocomplete demo

Lex the unfinished code with the same lexer as notebook 01, map tokens to ids, and let the model generate.

In [ ]:
import io, tokenize

def tokenize_code(source, partial=False):
    tokens = []
    prev_toktype = tokenize.INDENT
    try:
        for tok in tokenize.generate_tokens(io.StringIO(source).readline):
            tok_type, tok_str = tok.type, tok.string

            if tok_type in (tokenize.COMMENT, tokenize.ENDMARKER):
                continue

            # docstrings
            if tok_type == tokenize.STRING and prev_toktype in (
                tokenize.INDENT, tokenize.NEWLINE, tokenize.DEDENT
            ):
                prev_toktype = tok_type
                continue

            if tok_type == tokenize.INDENT:
                tokens.append("<INDENT>")
            elif tok_type == tokenize.DEDENT:
                tokens.append("<DEDENT>")
            elif tok_type in (tokenize.NEWLINE, tokenize.NL):
                # collapse blank lines / removed comments / removed docstrings
                if tokens and tokens[-1] not in ("<NEWLINE>", "<INDENT>", "<DEDENT>"):
                    tokens.append("<NEWLINE>")
            else:
                tokens.append(tok_str)
            prev_toktype = tok_type
        return tokens
    except (IndentationError, tokenize.TokenError, TabError):
        # partial=True keeps the tokens read so far (used for unfinished code at inference time)
        return tokens if partial else None

In [ ]:
def tokenize_prompt(code):
    tokens = tokenize_code(code, partial=True)

    # the lexer closes the last line on its own, but the user is still typing it
    if not code.endswith("\n"):
        while tokens and tokens[-1] in ("<NEWLINE>", "<DEDENT>"):
            tokens.pop()

    return tokens

In [ ]:
def complete(code):
    tokens = tokenize_prompt(code)[-SEQ_LEN:]
    ids = [BOS_IDX] + [vocab.get(t, vocab["<UNK>"]) for t in tokens]

    generated = model.generate(
        torch.tensor([ids], device=device),
        max_new_tokens=MAX_OUTPUT_LEN + 1,
        eos_idx=EOS_IDX
    )[0, len(ids):].tolist()

    return " ".join(id_to_token[i] for i in generated if i != EOS_IDX)

In [ ]:
prompts = [
    "def add(a, b):\n    return",
    "import numpy as",
    "for i in range(",
    "with open(path, 'r') as f:\n    data =",
    "if __name__ ==",
]

for prompt in prompts:
    print(prompt)
    print("  -->", complete(prompt))
    print()

## Model comparison

Fill in after running notebooks 03, 05 and 06 on Colab.

LSTM and GPT use the same vocab and the same test chunks, so all their numbers compare directly.
CodeT5 is evaluated on the raw-text pairs from `04_codet5_data` with its own tokenizer, so only
**exact line match** is a rough comparison across all three.

| Model | Params | Pretrained | Test loss | Token acc (teacher-forced) | Exact line match | Edit similarity |
|---|---|---|---|---|---|---|
| LSTM seq2seq | | no | | | | – |
| GPT (decoder-only) | | no | | | | – |
| CodeT5-small | ~60M | yes | (sub-word loss) | – | | |